# Train and test Burn-in DRQN on the simulated banks

EXP002: 通常 DRQN の学習ループに burn-in を導入。replay された系列の先頭 `burn_in_steps`
ステップを no-grad で LSTM に流して hidden state を構築し、その hidden を初期状態として
suffix 区間（step `burn_in_steps+1`〜40）のみで TD 損失を計算する。モデル構造は通常 DRQN のまま。

target Q の item mask は全履歴に基づき計算するため、burn-in より前に選ばれた項目も再選択されない。

設計の詳細は `EXP002/exp_summary.md` を参照。Colab 前提のため、リポジトリは
`MyDrive/Grad_Research_new` などに配置する。

In [ ]:
import copy
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
START_TOKEN = 2


def find_project_root():
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parents[3])

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "Grad_Research_new",
        cwd / "Grad_Research",
        cwd.parent,
        Path("/content/Grad_Research_new"),
        Path("/content/Grad_Research"),
        Path("/content/drive/MyDrive/Grad_Research_new"),
        Path("/content/drive/MyDrive/Grad_Research"),
        Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
    ])

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research_new or /content/Grad_Research_new."
    )


ROOT        = find_project_root()
EXP002_DIR  = ROOT / "EXP002"
MODEL_DIR   = EXP002_DIR / "models"
RESULTS_DIR = EXP002_DIR / "results"

print(f"Device      : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"Results dir : {RESULTS_DIR}")

In [ ]:
@dataclass
class Config:
    # DRQN
    embed_dim:    int   = 16
    lstm_hidden:  int   = 64
    dropout_rate: float = 0.0

    # Training
    test_length:         int   = 40
    gamma:               float = 0.1
    memory_capacity:     int   = 1000
    epsilon:             float = 0.1
    batch_size:          int   = 128
    q_network_iteration: int   = 40
    learning_rate:       float = 1e-3
    training_size:       int   = 1000
    validation_size:     int   = 200
    validation_interval: int   = 50

    # Bank / prior
    bank_type:   str = "uncor"   # "uncor" | "cor"
    bank_id:     int = 1
    prior:       str = "normal"  # "normal" | "uniform"
    n_items:     int = 200
    random_seed: int = 42

    # Burn-in
    burn_in_steps: int = 5

In [ ]:
from typing import Any, cast
from scipy.optimize import minimize_scalar


def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    resp = (np.random.rand(1) <= p).astype(int)
    return resp


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    info = (
        D**2 * a**2 * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )
    return info


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"))
    return np.array(result.x).reshape(1,)


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"))
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)

In [ ]:
# 通常 DRQN（src/models.py の DRQN と同等）。EXP002 はモデル構造を変えず学習ループのみ変更する。
class DRQN(nn.Module):
    def __init__(self, action_space, embed_dim, lstm_hidden, dropout_rate):
        super(DRQN, self).__init__()
        self.embed = nn.Embedding(3, embed_dim)  # 0=wrong, 1=correct, 2=start
        self.lstm = nn.LSTM(embed_dim, lstm_hidden, batch_first=True)
        self.out = nn.Linear(lstm_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)
        self.action_space = action_space
        self.embed_dim = embed_dim
        self.lstm_hidden = lstm_hidden

    def forward(self, resps, hidden=None):
        x = self.embed(resps)
        x = self.dropout(x)
        out, hidden = self.lstm(x, hidden)
        out = self.dropout(out)
        return self.out(out), hidden

    def init_hidden(self, batch_size=1):
        h = torch.zeros(1, batch_size, self.lstm_hidden, device=device)
        c = torch.zeros(1, batch_size, self.lstm_hidden, device=device)
        return (h, c)

    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if "weight" in name:
                        nn.init.kaiming_normal_(param)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def apply_positive_constraint(model, min_value=0.0):
    out_layer = cast(nn.Linear, model.out)
    out_layer.weight.data.clamp_(min=min_value)
    if out_layer.bias is not None:
        out_layer.bias.data.clamp_(min=min_value)

In [ ]:
def choose_action(model, prev_resp_t, hidden, item_id_arr, epsilon, action_space):
    with torch.no_grad():
        q_value, hidden = model(prev_resp_t, hidden)
        qv = q_value.squeeze(0).squeeze(0).clone()
        if item_id_arr.size > 0:
            qv[torch.from_numpy(item_id_arr).to(device)] = -float("inf")
        if np.random.randn() >= epsilon:
            action = int(qv.argmax().item())
        else:
            candidates = np.setdiff1d(np.arange(action_space), item_id_arr, assume_unique=False)
            action = int(np.random.choice(candidates))
    return action, hidden


def choose_action_test(model, prev_resps_t, hidden, item_id_history):
    with torch.no_grad():
        q_value, hidden = model(prev_resps_t, hidden)
        q_value = q_value.squeeze(1).cpu().numpy()
        if item_id_history.shape[0] > 0:
            row_index = np.tile(np.arange(item_id_history.shape[1])[np.newaxis, :], (item_id_history.shape[0], 1))
            q_value[row_index, item_id_history] = -np.inf
        action = q_value.argmax(axis=1)
    return action, hidden


def update_theta_state(item_bank, item_id_history, resp_history, theta_state):
    theta_next = np.zeros(theta_state.shape[0])
    idx_full = np.sum(resp_history, axis=0) == resp_history.shape[0]
    idx_zero = np.sum(resp_history, axis=0) == 0
    idx_norm = np.bitwise_not(idx_full | idx_zero)
    theta_next[idx_full] = theta_state[idx_full] + (item_bank[:, 1].max() - theta_state[idx_full]) / 2
    theta_next[idx_zero] = theta_state[idx_zero] + (item_bank[:, 1].min() - theta_state[idx_zero]) / 2
    if np.any(idx_norm):
        theta_next[idx_norm] = np.squeeze(
            MLE_TEST(item_bank[item_id_history[:, idx_norm]], resp_history[:, idx_norm])
        )
    return theta_next


def warm_hidden(model, resps_t, burn_in_steps):
    # 先頭 burn_in_steps トークンを no-grad で LSTM に流し、suffix 学習用の初期 hidden を作る
    if burn_in_steps <= 0:
        return model.init_hidden(resps_t.shape[0])

    warm_inputs = resps_t[:, :burn_in_steps]
    hidden = model.init_hidden(resps_t.shape[0])
    with torch.no_grad():
        _, hidden = model(warm_inputs, hidden)
    return tuple(state.detach() for state in hidden)

In [ ]:
def train(cfg, item_bank, action_space, eval_net, target_net):
    best_valid = None
    best_state = None

    loss_func = nn.MSELoss()
    optimizer = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)
    eval_net.train()

    memory = []
    memory_idx = 0
    learn_step_counter = 0

    if cfg.prior == "normal":
        training_theta = np.random.randn(cfg.training_size)
    elif cfg.prior == "uniform":
        training_theta = np.random.uniform(-3, 3, cfg.training_size)
    else:
        raise ValueError(f"Unsupported prior: {cfg.prior}")

    for j in range(cfg.training_size):
        prev_resp_t = torch.tensor([[START_TOKEN]], dtype=torch.long, device=device)
        hidden = eval_net.init_hidden(1)

        item_id_arr = np.array([], dtype=np.int64)
        resp_arr = np.array([], dtype=np.int64)
        theta_current = np.random.rand(1) - 0.5

        ep_resps = [START_TOKEN]
        ep_actions = []
        ep_rewards = []

        for _ in range(cfg.test_length):
            action, hidden = choose_action(
                eval_net, prev_resp_t, hidden, item_id_arr, cfg.epsilon, action_space
            )

            response = int(RESPOND(item_bank[np.array([action])], training_theta[j])[0])
            reward = FI(item_bank[np.array([action])], training_theta[j])

            item_id_arr = np.concatenate((item_id_arr, np.array([action], dtype=np.int64)))
            resp_arr = np.concatenate((resp_arr, np.array([response], dtype=np.int64)))

            if len(np.unique(resp_arr)) == 1:
                if response == 1:
                    theta_current = np.array(
                        [theta_current[-1] + (item_bank[:, 1].max() - theta_current[-1]) / 2]
                    )
                else:
                    theta_current = np.array(
                        [theta_current[-1] - (theta_current[-1] - item_bank[:, 1].min()) / 2]
                    )
            else:
                theta_current = MLE(item_bank[item_id_arr], resp_arr)

            ep_resps.append(response)
            ep_actions.append(action)
            ep_rewards.append(float(reward[0]))
            prev_resp_t = torch.tensor([[response]], dtype=torch.long, device=device)

        episode = {
            "resps": np.asarray(ep_resps, dtype=np.int64),
            "actions": np.asarray(ep_actions, dtype=np.int64),
            "rewards": np.asarray(ep_rewards, dtype=np.float32),
        }
        if len(memory) < cfg.memory_capacity:
            memory.append(episode)
        else:
            memory[memory_idx] = episode
        memory_idx = (memory_idx + 1) % cfg.memory_capacity

        if len(memory) >= cfg.batch_size:
            indices = np.random.choice(len(memory), cfg.batch_size, replace=False)
            batch = [memory[i] for i in indices]

            resps_t = torch.LongTensor(np.stack([ep["resps"] for ep in batch])).to(device)
            actions_t = torch.LongTensor(np.stack([ep["actions"] for ep in batch])).to(device)
            rewards_t = torch.FloatTensor(np.stack([ep["rewards"] for ep in batch])).to(device)

            if cfg.burn_in_steps >= cfg.test_length:
                raise ValueError("burn_in_steps must be smaller than test_length.")

            # burn-in: 先頭 burn_in_steps を no-grad で消費し、その hidden から suffix を学習
            hidden_eval = warm_hidden(eval_net, resps_t, cfg.burn_in_steps)
            suffix_resps = resps_t[:, cfg.burn_in_steps :]
            q_suffix_eval, _ = eval_net(suffix_resps, hidden_eval)
            q_eval = q_suffix_eval[:, :-1, :].gather(2, actions_t[:, cfg.burn_in_steps :].unsqueeze(-1)).squeeze(-1)

            with torch.no_grad():
                q_full_target, _ = target_net(resps_t)

            # mask は全履歴に基づき計算（burn-in 前に選ばれた項目も再選択不可）
            q_next_all = q_full_target[:, 1:, :].clone()
            selected_mask = torch.cumsum(F.one_hot(actions_t, num_classes=action_space), dim=1).bool()
            q_next_all[selected_mask] = -float("inf")
            q_next = q_next_all[:, cfg.burn_in_steps :, :].max(dim=2)[0]

            rewards_slice = rewards_t[:, cfg.burn_in_steps :]
            is_terminal = torch.zeros_like(rewards_t)
            is_terminal[:, -1] = 1.0
            terminal_slice = is_terminal[:, cfg.burn_in_steps :]
            q_target = rewards_slice + cfg.gamma * q_next * (1.0 - terminal_slice)

            loss = loss_func(q_eval, q_target)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(eval_net.parameters(), max_norm=1.0)
            optimizer.step()
            apply_positive_constraint(eval_net)

            learn_step_counter += 1
            if learn_step_counter % cfg.q_network_iteration == 0:
                target_net.load_state_dict(eval_net.state_dict())

        ### Validation ###
        if (j + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_bias = np.zeros((cfg.test_length, cfg.validation_size))
            valid_theta = np.random.choice(training_theta, cfg.validation_size)
            theta_state = np.random.rand(cfg.validation_size) - 0.5

            prev_resps_t = torch.full((cfg.validation_size, 1), START_TOKEN, dtype=torch.long, device=device)
            valid_hidden = eval_net.init_hidden(cfg.validation_size)
            item_id_history = np.empty((0, cfg.validation_size), dtype=np.int64)
            resp_history = np.empty((0, cfg.validation_size), dtype=np.int64)

            for i in range(cfg.test_length):
                action, valid_hidden = choose_action_test(eval_net, prev_resps_t, valid_hidden, item_id_history)
                response = RESPOND(item_bank[action], valid_theta).astype(np.int64)

                item_id_history = np.concatenate((item_id_history, action[np.newaxis, :]))
                resp_history = np.concatenate((resp_history, response[np.newaxis, :]))
                theta_state = update_theta_state(item_bank, item_id_history, resp_history, theta_state)
                valid_bias[i] = theta_state - valid_theta
                prev_resps_t = torch.from_numpy(response).long().unsqueeze(1).to(device)

            step_valid = np.transpose(
                np.vstack(
                    (
                        np.arange(1, cfg.test_length + 1),
                        np.mean(valid_bias, axis=1),
                        np.sqrt(np.mean(valid_bias**2, axis=1)),
                        np.mean(np.abs(valid_bias), axis=1),
                    )
                )
            )
            print(f"subject: {j + 1}\n\n{step_valid}\n")

            result_valid = np.mean(step_valid[6:, 1:], axis=0)
            if best_valid is None:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())
            elif (abs(result_valid[0]) < abs(best_valid[0])) and np.sum(result_valid[1:] < best_valid[1:]) == 2:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())

            eval_net.train()

    return best_state

In [ ]:
def test(cfg, item_bank, theta_test, eval_net):
    with torch.no_grad():
        eval_net.eval()
        testing_size = len(theta_test)
        theta_state = np.random.rand(testing_size) - 0.5

        prev_resps_t = torch.full((testing_size, 1), START_TOKEN, dtype=torch.long, device=device)
        test_hidden = eval_net.init_hidden(testing_size)

        item_id_history = np.empty((0, testing_size), dtype=np.int64)
        resp_history = np.empty((0, testing_size), dtype=np.int64)
        step_rows = []
        theta_all = []

        for i in range(cfg.test_length):
            action, test_hidden = choose_action_test(eval_net, prev_resps_t, test_hidden, item_id_history)
            response = RESPOND(item_bank[action], theta_test).astype(np.int64)

            item_id_history = np.concatenate((item_id_history, action[np.newaxis, :]))
            resp_history = np.concatenate((resp_history, response[np.newaxis, :]))
            theta_state = update_theta_state(item_bank, item_id_history, resp_history, theta_state)
            theta_all.append(theta_state.copy())

            bias = theta_state - theta_test
            row = [i + 1, np.mean(bias), np.sqrt(np.mean(bias**2)), np.mean(np.abs(bias))]
            step_rows.append(row)
            print("step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(*row))

            prev_resps_t = torch.from_numpy(response).long().unsqueeze(1).to(device)

    theta_matrix = np.stack(theta_all, axis=0)
    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length).reshape(-1, 1)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size).reshape(-1, 1)
    item_id_col = (item_id_history + 1).transpose().reshape(-1, 1)
    resp_col = resp_history.transpose().reshape(-1, 1)
    theta_est_col = theta_matrix.transpose().reshape(-1, 1)
    bias_col = (theta_matrix - theta_test).transpose().reshape(-1, 1)

    records = pd.DataFrame(
        np.hstack([user_id_col, step_col, item_id_col, resp_col, theta_est_col, bias_col]),
        columns=["userID", "step", "itemID", "resp", "theta_est", "bias"],
    )
    summary = pd.DataFrame(step_rows, columns=["step", "Bias", "RMSE", "MAE"])

    stem = f"{cfg.bank_type}_{cfg.bank_id}_DRQN_burnin_{cfg.burn_in_steps}_{cfg.prior}_gamma_{cfg.gamma}"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    records.to_csv(RESULTS_DIR / f"records_{stem}.csv", index=False)
    summary.to_csv(RESULTS_DIR / f"summary_{stem}.csv", index=False)
    print(f"\nSaved to {RESULTS_DIR}")

In [ ]:
cfg = Config()
set_seed(cfg.random_seed)

bank_dir = {
    "uncor": ROOT / "data" / "uncorrelated_banks",
    "cor":   ROOT / "data" / "correlated_banks",
}[cfg.bank_type]

item_bank    = np.array(pd.read_csv(bank_dir / f"item_bank_{cfg.bank_type}_{cfg.bank_id}.csv")[["a", "b", "c"]])[:cfg.n_items]
action_space = item_bank.shape[0]
theta_test   = np.array(pd.read_csv(ROOT / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv")["x"])

print(f"item bank  : {item_bank.shape}")
print(f"theta_test : {theta_test.shape}")
print(f"\nConfig:\n{cfg}")

In [ ]:
eval_net   = DRQN(action_space, cfg.embed_dim, cfg.lstm_hidden, cfg.dropout_rate).to(device)
target_net = DRQN(action_space, cfg.embed_dim, cfg.lstm_hidden, cfg.dropout_rate).to(device)
eval_net.initialize()
target_net.initialize()
target_net.load_state_dict(eval_net.state_dict())

best_state = train(cfg, item_bank, action_space, eval_net, target_net)

assert best_state is not None, "No checkpoint was saved. Increase training_size or lower validation_interval."
eval_net.load_state_dict(best_state)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / f"drqn_burnin_{cfg.burn_in_steps}_{cfg.prior}_{cfg.bank_type}_{cfg.bank_id}_gamma_{cfg.gamma}.pt"
torch.save(eval_net.state_dict(), model_path)
print(f"Model saved to: {model_path}")

test(cfg, item_bank, theta_test, eval_net)

In [ ]:
# 保存した state_dict を読み込む場合は DRQN を再構築してから load する
eval_net = DRQN(action_space, cfg.embed_dim, cfg.lstm_hidden, cfg.dropout_rate).to(device)
eval_net.load_state_dict(torch.load(
    MODEL_DIR / f"drqn_burnin_{cfg.burn_in_steps}_{cfg.prior}_{cfg.bank_type}_{cfg.bank_id}_gamma_{cfg.gamma}.pt",
    map_location=device,
))

test(cfg, item_bank, theta_test, eval_net)